# End of week 1 exercise

To demonstrate your familiarity with your AI API choice (Z.AI), OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display, update_display

In [2]:
# constants

MODEL_GPT = 'glm-5'
MODEL_LLAMA = 'qwen-2.5-coder'

In [3]:
# set up environment

load_dotenv(override=True)
api_key = os.getenv('ZAI_API_KEY')

# Z.AI client for GLM models
client = OpenAI(
    api_key=api_key,
    base_url="https://api.z.ai/api/paas/v4/"
)

# Ollama client for local open-source models
ollama_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

In [4]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [5]:
# Get glm-5 to answer, with streaming

system_prompt = "You are a helpful technical assistant that explains code clearly and concisely."

stream = client.chat.completions.create(
    model=MODEL_GPT,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question}
    ],
    stream=True
)

response = ""
display_handle = display(Markdown(""), display_id=True)
for chunk in stream:
    response += chunk.choices[0].delta.content or ""
    update_display(Markdown(response), display_id=display_handle.display_id)

This line of code defines a generator that yields **unique author names** from a collection of books, ignoring any books that are missing author information.

Here is the breakdown of how it works, from the inside out:

### 1. The Set Comprehension
`{book.get("author") for book in books if book.get("author")}`

This part creates a **set** (a collection of unique items) based on three operations:
*   **Iteration:** It loops through every `book` in the `books` list.
*   **Filtering (`if`):** It checks if the book has a valid "author" key. Using `.get()` is safe; if the key is missing, it returns `None` (or a default), which the `if` statement treats as False, effectively skipping that book.
*   **Extraction:** It retrieves the value of "author".
*   **Deduplication (`{}`):** Because this is a set comprehension (denoted by curly braces `{}`), **duplicates are automatically removed**. If five books have "Jane Austen", the set will only contain "Jane Austen" once.

### 2. The Yield Statement
`yield from ...`

*   This turns the expression into a generator.
*   `yield from` takes the set created in step 1 and iterates over it, yielding one author at a time to the caller.

### Summary of Purpose
The code efficiently performs three tasks simultaneously:
1.  **Extracts** data (authors).
2.  **Filters** out missing data (books without authors).
3.  **Deduplicates** the results (ensures each author is listed only once).

### Example
If `books` looks like this:
```python
books = [
    {"title": "Book A", "author": "Alice"},
    {"title": "Book B", "author": "Bob"},
    {"title": "Book C", "author": "Alice"}, # Duplicate author
    {"title": "Book D"}                     # Missing author
]
```

The code will yield:
1.  `"Alice"`
2.  `"Bob"`
*(Note: The order is not guaranteed because sets are unordered collections.)*

In [8]:
# Get qwen-2.5-coder to answer (running locally via Ollama)

response = ollama_client.chat.completions.create(
    model=MODEL_LLAMA,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question}
    ]
)

display(Markdown(response.choices[0].message.content))

```<|im_start|>
<|im_start|> This Python code uses the `yield from` statement to iterate over a set comprehension that extracts unique author names from a list of book dictionaries. Let's break down the code step-by-step:

1. `{book.get("author") for book in books if book.get("author")}`
   - This is a set comprehension.
   - It iterates over each `book` in the `books` list.
   - For each `book`, it attempts to retrieve the value associated with the key `"author"`.
   - The expression `book.get("author")` returns the author's name if it exists, and `None` otherwise.
   - The comprehension includes only those values that are not `None` (i.e., valid author names).

2. `yield from {book.get("author") for book in books if book.get("author")}`
   - This uses the `yield from` statement to delegate control to the set comprehension.
   - When an item is produced by the set comprehension, it is yielded back to the caller of `yield from`.

Summary:
- The code yields each unique author name found in the `books` list, skipping any entries where an author name is not present or is invalid (`None`).